In [6]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "windowing_filtered.py").exists():
    sys.path.insert(0, str(NOTEBOOK_DIR))
elif (NOTEBOOK_DIR / "preprocessing" / "windowing_filtered.py").exists():
    sys.path.insert(0, str(NOTEBOOK_DIR / "preprocessing"))

import pandas as pd
import numpy as np
from windowing_filtered import create_windows_filtered

# Empatica
eda_e4   = pd.read_csv('../Data/preprocessed_data/filtered/eda_filtered.csv')['EDA'].values
bvp_e4   = pd.read_csv('../Data/preprocessed_data/filtered/bvp_filtered.csv')['BVP'].values
hr_e4    = pd.read_csv('../Data/preprocessed_data/filtered/hr_filtered.csv')['HR'].values
temp_e4  = pd.read_csv('../Data/preprocessed_data/filtered/temp_filtered.csv')['TEMP'].values

# RespiBAN
ecg_resp  = pd.read_csv('../Data/preprocessed_data/filtered/ecg_filtered_respiban.csv')['ECG'].values
eda_resp  = pd.read_csv('../Data/preprocessed_data/filtered/eda_filtered_respiban.csv')['EDA'].values
emg_resp  = pd.read_csv('../Data/preprocessed_data/filtered/emg_centered_respiban.csv')['EMG'].values
resp_resp = pd.read_csv('../Data/preprocessed_data/filtered/resp_filtered_respiban.csv')['RESP'].values
temp_resp = pd.read_csv('../Data/preprocessed_data/filtered/temp_filtered_respiban.csv')['TEMP'].values

print(len(eda_e4), len(bvp_e4), len(hr_e4), len(temp_e4))
print(len(ecg_resp), len(eda_resp), len(emg_resp), len(resp_resp), len(temp_resp))

31494 503943 7715 4442067
4442067 4442067 4442067 4442067 4442067


In [7]:
signals = {
    'EDA_E4':  eda_e4,
    'BVP_E4':  bvp_e4,
    'HR_E4':   hr_e4,
    'TEMP_E4': temp_e4,
    'ECG_RB':  ecg_resp,
    'EDA_RB':  eda_resp,
    'EMG_RB':  emg_resp,
    'RESP_RB': resp_resp,
    'TEMP_RB': temp_resp,
}

fs_dict = {
    'EDA_E4':  4,
    'BVP_E4':  64,
    'HR_E4':   1,
    'TEMP_E4': 4,
    'ECG_RB':  700,
    'EDA_RB':  700,
    'EMG_RB':  700,
    'RESP_RB': 700,
    'TEMP_RB': 700,
}

In [8]:
result = create_windows_filtered(signals, fs_dict, window_sec=5, overlap=0.5)

for name, arr in result['windows'].items():
    print(name, arr.shape)

EDA_E4 (3148, 20)
BVP_E4 (3148, 320)
HR_E4 (3856, 5)
TEMP_E4 (444205, 20)
ECG_RB (2537, 3500)
EDA_RB (2537, 3500)
EMG_RB (2537, 3500)
RESP_RB (2537, 3500)
TEMP_RB (2537, 3500)


In [12]:
import numpy as np
import pandas as pd

ibi_values = pd.read_csv('../Data/preprocessed_data/filtered/ibi_filtered.csv')['IBI'].values

# reconstruct approximate timestamp of each beat by cumulatively summing the intervals
timestamps = np.cumsum(ibi_values)

ibi_df = pd.DataFrame({'time': timestamps, 'IBI': ibi_values})
print(ibi_df.head())
print('Total duration:', timestamps[-1], 'seconds')

# now window by time
window_sec, overlap = 5, 0.5
step_sec = window_sec * (1 - overlap)

ibi_windows = []
t_start = ibi_df['time'].iloc[0]
t_end = ibi_df['time'].iloc[-1]

t = t_start
while t + window_sec <= t_end:
    mask = (ibi_df['time'] >= t) & (ibi_df['time'] < t + window_sec)
    ibi_windows.append(ibi_df.loc[mask, 'IBI'].values)
    t += step_sec

print(len(ibi_windows), 'IBI windows')
print('window 0 has', len(ibi_windows[0]), 'beats:', ibi_windows[0])
print('window 1 has', len(ibi_windows[1]), 'beats:', ibi_windows[1])

       time       IBI
0  0.781286  0.781286
1  1.578197  0.796911
2  2.359484  0.781286
3  3.140770  0.781286
4  3.922056  0.781286
Total duration: 2953.5412694999186 seconds
1180 IBI windows
window 0 has 7 beats: [0.781286  0.7969115 0.781286  0.781286  0.781286  0.781286  0.750034 ]
window 1 has 6 beats: [0.781286 0.781286 0.750034 0.750034 0.750034 0.796911]


In [10]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "windowing_filtered.py").exists():
    sys.path.insert(0, str(NOTEBOOK_DIR))
elif (NOTEBOOK_DIR / "preprocessing" / "windowing_filtered.py").exists():
    sys.path.insert(0, str(NOTEBOOK_DIR / "preprocessing"))

import os
from windowing_filtered import windows_to_dataframe

for name in signals:
    df = windows_to_dataframe(result, name)
    df.to_csv(f'../Data/preprocessed_data/windows_filtered/windows_{name}.csv', index=False)

print("Saved:", [f for f in os.listdir() if f.startswith('windows_')])

Saved: ['windows_BVP_E4.csv', 'windows_ECG_RB.csv', 'windows_EDA_E4.csv', 'windows_EDA_RB.csv', 'windows_EMG_RB.csv', 'windows_HR_E4.csv', 'windows_RESP_RB.csv', 'windows_TEMP_E4.csv', 'windows_TEMP_RB.csv']


In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "windowing_filtered.py").exists():
    sys.path.insert(0, str(NOTEBOOK_DIR))
elif (NOTEBOOK_DIR / "preprocessing" / "windowing_filtered.py").exists():
    sys.path.insert(0, str(NOTEBOOK_DIR / "preprocessing"))

from windowing_filtered import plot_all_windowed_signals
import os

figure_dir = os.path.join('..', 'outputs', '04_windowing_filtered', 'figures')
saved_plots = plot_all_windowed_signals(
    result,
    output_dir=figure_dir,
    prefix='filtered',
    max_windows=5,
    show=False,
)

print('Saved filtered window plots:')
for path in saved_plots:
    print(path)
